In [3]:
import datetime
import requests
import pandas as pd
import pika
from lxml import html
import openpyxl
from io import BytesIO
import json

In [4]:
connection = pika.BlockingConnection(pika.ConnectionParameters(host='localhost'))
channel = connection.channel()

In [ ]:
def post_to_rabbitmq(ROUTING_KEY:str, data):
    channel.basic_publish(exchange='mutual_fund_data_collection', routing_key=ROUTING_KEY, body=data)

In [ ]:
class Fetch_List_of_AMC:
    def __init__(self) -> None:
        self.AMFI_WEBPAGE_URL="https://www.amfiindia.com/listofgroupcompanynames1"
        self.ROUTING_KEY="list_of_amc"
        
    
    def fetch_amc_csv_hyperlink(self):
        response = requests.get(self.AMFI_WEBPAGE_URL)
        tree = html.fromstring(response.content)
        element = tree.xpath('//*[@id="tab2"]/h2/a[1]')[0]
        # print(element)
        hyperlink = element.attrib['href']
        return hyperlink
    
    def fetch_amc_csv_data(self, hyperlink):
        create_necessary_url = lambda hyperlink: "https://www.amfiindia.com"+hyperlink
        response = requests.get(create_necessary_url(hyperlink))
        return response
    
    def parse_amc_data(self, response):
        amc_data=[]
        workbook = openpyxl.load_workbook(filename = BytesIO(response.content))
        worksheet = workbook.active
        for row in worksheet.iter_rows(values_only=True):
            amc_data.append(row)
            
        amc_data = pd.DataFrame(amc_data)
        new_header = amc_data.iloc[0]
        amc_data = amc_data[1:]
        amc_data.columns = new_header
        return amc_data
    
    def runner_module(self):
        hyperlink=amc_list.fetch_amc_csv_hyperlink()
        amc_data=amc_list.fetch_amc_csv_data(hyperlink)
        final_amc_data=self.parse_amc_data(amc_data)
        print("Sending Data Message")
        channel.basic_publish(exchange='mutual_fund_data_collection', routing_key=self.ROUTING_KEY, body=final_amc_data.to_json(orient='index'))
        return final_amc_data

amc_list=Fetch_List_of_AMC()
# amc_list.runner_module().head()

In [ ]:
# import time
# while True:
#     time.sleep(5)
#     amc_list.runner_module().head()

In [ ]:
class Fetch_List_of_Mutual_Fund:
    def __init__(self) -> None:
        self.AMFI_WEBPAGE_URL="https://portal.amfiindia.com/DownloadSchemeData_Po.aspx?mf=0"
        self.ROUTING_KEY="list_of_mutual_fund"
    
    def fetch_list_of_mutual_fund_data(self):
        mf_data= pd.read_csv(mf_list.AMFI_WEBPAGE_URL)
        return mf_data
    
    def clean_isin(self, isin_value):
            # print("Current ISIN Value:", isin_value)
            isNaN = lambda num: num!=num
            isin_segregated_data={
                "status":-1,
                "ISIN Div Payout/ ISIN Growth":"",
                "ISIN Div Reinvestment":""
            }
            if isNaN(isin_value):
                isin_segregated_data["status"]=0
            else:
                spilt_isin=isin_value.split("INF")
                # print("Splited ISIN:", spilt_isin)
                if len(spilt_isin)==2:
                    isin_segregated_data["status"]=1
                    isin_segregated_data["ISIN Div Payout/ ISIN Growth"]="INF"+spilt_isin[1].strip()
                elif len(spilt_isin)==3:
                    isin_segregated_data["status"]=2
                    isin_segregated_data["ISIN Div Payout/ ISIN Growth"]="INF"+spilt_isin[1].strip()
                    isin_segregated_data["ISIN Div Reinvestment"]="INF"+spilt_isin[2].strip()
            return isin_segregated_data

    def filtering_data(self, data):
        ISIN_COLUMN="ISIN Div Payout/ ISIN GrowthISIN Div Reinvestment"
        # isin_values=isin_data[ISIN_COLUMN].values
        
        filtered_isin=[]
        for i in range(0,len(data)):
            CURRENT_ISIN_VALUE=data[ISIN_COLUMN].iloc[i]
            filtered_isin.append(self.clean_isin(CURRENT_ISIN_VALUE))
        isin_data=pd.concat([data, pd.json_normalize(filtered_isin)], axis=1)
        del isin_data['status']
        del isin_data[ISIN_COLUMN]
        return isin_data    
        
    def runner_module(self):
        data=self.fetch_list_of_mutual_fund_data()
        isin_data=mf_list.filtering_data(data)
        isin_data=isin_data[["AMC","Code","Scheme Name","Scheme NAV Name",'Scheme Category',
                            'ISIN Div Payout/ ISIN Growth', 'ISIN Div Reinvestment']]
        
        # channel.basic_publish(exchange='mutual_fund_data_collection', routing_key=self.ROUTING_KEY, body=isin_data.to_dict()) 
        return isin_data
mf_list=Fetch_List_of_Mutual_Fund()

In [ ]:
mf_list.runner_module().head()

In [6]:
class Nav_Value_Fetch_Particular_Date:
    def __init__(self, fetch_date) -> None:
        self.AMFI_WEBPAGE_URL="https://www.amfiindia.com/modules/NavHistoryAll"
        self.fetch_date=fetch_date
        self.ROUTING_KEY="daily_nav_data"
        
    def nav_data_fetch(self):
        create_url_necessary_date = lambda current_date, URl: self.AMFI_WEBPAGE_URL+'?'+"fDate="+current_date.strftime('%Y-%b-%d')
        response = requests.post(create_url_necessary_date(self.fetch_date, self.AMFI_WEBPAGE_URL))    
        return response
    
    def parse_data_to_pandas(self, response):
        data=pd.read_html(response.content,header=0)
        data=pd.DataFrame(data[0])
        isNaN = lambda num: num!=num
        nav_daily_data=[]
        for i in range(1,len(data)):
            CURRENT_DATA=data.iloc[i]
            # print(type(CURRENT_DATA.iloc[1]))
            if CURRENT_DATA.iloc[0]==CURRENT_DATA.iloc[1]==CURRENT_DATA.iloc[2]==CURRENT_DATA.iloc[3]:
                if CURRENT_DATA.iloc[0].__contains__("Mutual Fund") :
                    AMC_CATEGORY=CURRENT_DATA.iloc[0]
                else:
                    FUND_NAME=CURRENT_DATA.iloc[0]
            if isNaN(CURRENT_DATA.iloc[1]) and isNaN(CURRENT_DATA.iloc[2]):
                NAV_VALUE=CURRENT_DATA.iloc[0]
                TIMESTAMP=CURRENT_DATA.iloc[3]
                nav_daily_data.append([AMC_CATEGORY,FUND_NAME, NAV_VALUE, TIMESTAMP])
                
        nav_daily_data=pd.DataFrame(nav_daily_data, columns=["AMC Fund Company", "Mutual Fund Name", "NAV Value", "NAV Record Timestamp"])
        nav_daily_data.reset_index(inplace=True)
        nav_daily_data['Timestamp']=datetime.datetime.now()
        nav_daily_data.rename(columns={ "index":"id",
                                "AMC Fund Company": "amc_mutual_fund", 
                                "Mutual Fund Name":"mutual_fund_name",
                                "NAV Value":"nav",
                                "Timestamp":"timestamp",
                                "NAV Record Timestamp":"nav_upload_date_time"}, inplace=True)
        return nav_daily_data
    
    def empty_data_check(self, response):
        if str(response.content).__contains__('No records to display'):
            return True
        else:
            return False
        
    def runner_module(self):
        response=self.nav_data_fetch()
        if self.empty_data_check(response):
            return -1
        else:
            nav_daily_data=self.parse_data_to_pandas(response)
            channel.basic_publish(exchange='mutual_fund_data_collection', routing_key=self.ROUTING_KEY, body=(nav_daily_data.to_json(orient='index')))
            return nav_daily_data

current_date=datetime.datetime.today()-datetime.timedelta(days=1)
print("Fetching Data for date:", current_date)
nav_data_particular_date=Nav_Value_Fetch_Particular_Date(current_date)
current_data=nav_data_particular_date.runner_module()

In [8]:
response_pattern={}
for i in range(1, 41):
    current_date=datetime.datetime.today()-datetime.timedelta(days=i)
    print("Fetching Data for date:", current_date)
    nav_data_particular_date=Nav_Value_Fetch_Particular_Date(current_date)
    current_data=nav_data_particular_date.runner_module()
    if type(current_data) == int:
        print("Unable to retrieve Data for date:", current_date)
        response_pattern[current_date]=0
    else:
        print("Data Fetch Successfull for date:", current_date)
        response_pattern[current_date]=1

Fetching Data for date: 2023-05-04 02:49:09.353158
Data Fetch Successfull for date: 2023-05-04 02:49:09.353158
Fetching Data for date: 2023-05-03 02:49:28.119102
Data Fetch Successfull for date: 2023-05-03 02:49:28.119102
Fetching Data for date: 2023-05-02 02:49:48.632753
Data Fetch Successfull for date: 2023-05-02 02:49:48.632753
Fetching Data for date: 2023-05-01 02:50:10.466575
Data Fetch Successfull for date: 2023-05-01 02:50:10.466575
Fetching Data for date: 2023-04-30 02:50:11.699080
Data Fetch Successfull for date: 2023-04-30 02:50:11.699080
Fetching Data for date: 2023-04-29 02:50:12.610902
Data Fetch Successfull for date: 2023-04-29 02:50:12.610902
Fetching Data for date: 2023-04-28 02:50:13.533749
Data Fetch Successfull for date: 2023-04-28 02:50:13.533749
Fetching Data for date: 2023-04-27 02:50:34.786512
Data Fetch Successfull for date: 2023-04-27 02:50:34.786512
Fetching Data for date: 2023-04-26 02:50:54.383038
Data Fetch Successfull for date: 2023-04-26 02:50:54.383038
F

In [9]:
response_pattern

{datetime.datetime(2023, 5, 4, 2, 49, 9, 353158): 1,
 datetime.datetime(2023, 5, 3, 2, 49, 28, 119102): 1,
 datetime.datetime(2023, 5, 2, 2, 49, 48, 632753): 1,
 datetime.datetime(2023, 5, 1, 2, 50, 10, 466575): 1,
 datetime.datetime(2023, 4, 30, 2, 50, 11, 699080): 1,
 datetime.datetime(2023, 4, 29, 2, 50, 12, 610902): 1,
 datetime.datetime(2023, 4, 28, 2, 50, 13, 533749): 1,
 datetime.datetime(2023, 4, 27, 2, 50, 34, 786512): 1,
 datetime.datetime(2023, 4, 26, 2, 50, 54, 383038): 1,
 datetime.datetime(2023, 4, 25, 2, 51, 13, 600266): 1,
 datetime.datetime(2023, 4, 24, 2, 51, 35, 840338): 1,
 datetime.datetime(2023, 4, 23, 2, 51, 57, 503173): 1,
 datetime.datetime(2023, 4, 22, 2, 51, 58, 793688): 1,
 datetime.datetime(2023, 4, 21, 2, 51, 59, 796629): 1,
 datetime.datetime(2023, 4, 20, 2, 52, 18, 918907): 1,
 datetime.datetime(2023, 4, 19, 2, 52, 39, 970151): 1,
 datetime.datetime(2023, 4, 18, 2, 52, 58, 352079): 1,
 datetime.datetime(2023, 4, 17, 2, 53, 17, 544786): 1,
 datetime.datet

In [ ]:
# response=nav_data_particular_date.nav_data_fetch()
# parsed_data=nav_data_particular_date.parse_data_to_pandas(response)

In [ ]:
# import time
# while True:
#     time.sleep(5)
#     print("Sending Data")
#     post_to_rabbitmq('daily_nav_data',current_data.to_json(orient='index'))

In [ ]:
# connection = pika.BlockingConnection(pika.ConnectionParameters(host='localhost'))
# channel = connection.channel()

In [ ]:
# post_to_rabbitmq('daily_nav_data',current_data.to_json(orient='index'))

In [ ]:
connection.close()